<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX Falcon 9 : prédiction de l'atterrissage du premier étage**

## Lab pratique : compléter le lab de prédiction par apprentissage automatique

Temps estimé : **60** minutes

SpaceX annonce sur son site des lancements de Falcon 9 à 62 millions de dollars, alors que les autres fournisseurs facturent au moins 165 millions de dollars chacun ; l'essentiel de l'économie vient de la réutilisation du premier étage par SpaceX. Par conséquent, si nous pouvons déterminer si le premier étage atterrira, nous pouvons déterminer le coût d'un lancement. Cette information peut servir à une entreprise concurrente qui souhaite se positionner face à SpaceX pour un lancement. Dans ce lab, vous allez créer un pipeline de machine learning pour prédire si le premier étage atterrira, à partir des données des labs précédents.

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Voici plusieurs exemples d'atterrissages ratés :

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


La plupart des atterrissages ratés sont planifiés : SpaceX effectue un atterrissage contrôlé dans l'océan.

## Objectifs

Réaliser une analyse exploratoire des données et déterminer les étiquettes d'entraînement

*   créer une colonne pour la classe
*   standardiser les données
*   séparer les données en données d'entraînement et données de test

\-Trouver les meilleurs hyperparamètres pour le SVM, les arbres de classification et la régression logistique

*   Déterminer la méthode la plus performante sur les données de test

## Importer les bibliothèques et définir les fonctions auxiliaires

In [1]:
import piplite
await piplite.install(['numpy'])
await piplite.install(['pandas'])
await piplite.install(['seaborn'])

Nous allons importer les bibliothèques suivantes pour ce lab

In [ ]:
# Pandas : bibliothèque Python de manipulation et d'analyse de données.
import pandas as pd
# NumPy : bibliothèque Python qui ajoute les tableaux multidimensionnels et de nombreuses fonctions mathématiques.
import numpy as np
# Matplotlib : bibliothèque de tracé ; pyplot offre une interface de tracé de type MATLAB.
import matplotlib.pyplot as plt
# Seaborn : bibliothèque de visualisation statistique basée sur matplotlib, avec une interface de haut niveau.
import seaborn as sns
# Preprocessing permet de standardiser nos données
from sklearn import preprocessing
# Permet de séparer nos données en données d'entraînement et de test
from sklearn.model_selection import train_test_split
# Permet de tester les paramètres des algorithmes de classification et de trouver le meilleur
from sklearn.model_selection import GridSearchCV
# Algorithme de classification : régression logistique
from sklearn.linear_model import LogisticRegression
# Algorithme de classification : machine à vecteurs de support (SVM)
from sklearn.svm import SVC
# Algorithme de classification : arbre de décision
from sklearn.tree import DecisionTreeClassifier
# Algorithme de classification : k plus proches voisins
from sklearn.neighbors import KNeighborsClassifier

Cette fonction trace la matrice de confusion.

In [ ]:
def plot_confusion_matrix(y,y_predict):
    "cette fonction trace la matrice de confusion"
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y, y_predict)
    ax= plt.subplot()
    sns.heatmap(cm, annot=True, ax = ax); # annot=True pour annoter les cellules
    ax.set_xlabel('Étiquettes prédites')
    ax.set_ylabel('Étiquettes réelles')
    ax.set_title('Matrice de confusion');
    ax.xaxis.set_ticklabels(["n'a pas atterri", 'a atterri']); ax.yaxis.set_ticklabels(["n'a pas atterri", 'a atterri'])
    plt.show() 

## Charger le dataframe

Charger les données

In [ ]:
from js import fetch
import io

URL1 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
resp1 = await fetch(URL1)
text1 = io.BytesIO((await resp1.arrayBuffer()).to_py())
data = pd.read_csv(text1)

In [ ]:
data.head()

In [ ]:
URL2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'
resp2 = await fetch(URL2)
text2 = io.BytesIO((await resp2.arrayBuffer()).to_py())
X = pd.read_csv(text2)

In [ ]:
X.head(100)

## TÂCHE 1

Créez un tableau NumPy à partir de la colonne <code>Class</code> de <code>data</code> en appliquant la méthode <code>to_numpy()</code>, puis affectez-le à la variable <code>Y</code> ; veillez à n'utiliser qu'une seule paire de crochets (df\['nom de la colonne']) pour sélectionner la colonne.

In [ ]:
Y = data['Class'].to_numpy()

## TÂCHE 2

Standardisez les données de <code>X</code>, puis réaffectez le résultat à la variable <code>X</code> à l'aide de la transformation fournie ci-dessous.

In [ ]:
# Fourni par l'énoncé
transform = preprocessing.StandardScaler()
# Standardiser X (moyenne 0, écart-type 1) et réaffecter le résultat à X
X = transform.fit_transform(X)

Nous séparons les données en données d'entraînement et données de test avec la fonction <code>train_test_split</code>. Les données d'entraînement sont ensuite divisées en données de validation (un second jeu servant à l'entraînement) ; les modèles sont alors entraînés et les hyperparamètres sélectionnés avec la fonction <code>GridSearchCV</code>.

## TÂCHE 3

Utilisez la fonction train_test_split pour séparer les données X et Y en données d'entraînement et de test. Fixez le paramètre test_size à 0.2 et random_state à 2. Les données d'entraînement et de test doivent être affectées aux variables suivantes.

<code>X_train, X_test, Y_train, Y_test</code>


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

Nous voyons que nous n'avons que 18 échantillons de test.

In [ ]:
Y_test.shape

## TÂCHE 4

Créez un objet de régression logistique, puis créez un objet <code>GridSearchCV</code> nommé <code>logreg_cv</code> avec cv = 10. Ajustez (fit) l'objet pour trouver les meilleurs paramètres du dictionnaire <code>parameters</code>.

In [ ]:
parameters ={'C':[0.01,0.1,1],
             'penalty':['l2'],
             'solver':['lbfgs']}

In [ ]:
parameters ={"C":[0.01,0.1,1],'penalty':['l2'], 'solver':['lbfgs']}# l1 lasso, l2 ridge
lr=LogisticRegression()

logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, Y_train)

Nous affichons l'objet <code>GridSearchCV</code> de la régression logistique. Nous affichons les meilleurs paramètres avec l'attribut <code>best_params\_</code> et la précision (accuracy) sur les données de validation avec l'attribut <code>best_score\_</code>.

In [ ]:
print("hyperparamètres optimisés (meilleurs paramètres) :",logreg_cv.best_params_)
print("précision (accuracy) :",logreg_cv.best_score_)

## TÂCHE 5

Calculez la précision (accuracy) sur les données de test avec la méthode <code>score</code> :

In [ ]:
logreg_cv.score(X_test, Y_test)

Regardons la matrice de confusion :

In [ ]:
yhat=logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test,yhat)

En examinant la matrice de confusion, nous voyons que la régression logistique sait distinguer les différentes classes. Le problème vient des faux positifs.

Résumé :

Vrais positifs - 12 (l'étiquette réelle est « a atterri », l'étiquette prédite aussi)

Faux positifs - 3 (l'étiquette réelle est « n'a pas atterri », l'étiquette prédite est « a atterri »)

## TÂCHE 6

Créez un objet machine à vecteurs de support (SVM), puis créez un objet <code>GridSearchCV</code> nommé <code>svm_cv</code> avec cv = 10. Ajustez (fit) l'objet pour trouver les meilleurs paramètres du dictionnaire <code>parameters</code>.

In [ ]:
parameters = {'kernel':('linear', 'rbf','poly','rbf', 'sigmoid'),
              'C': np.logspace(-3, 3, 5),
              'gamma':np.logspace(-3, 3, 5)}
svm = SVC()

In [ ]:
svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, Y_train)

In [ ]:
print("hyperparamètres optimisés (meilleurs paramètres) :",svm_cv.best_params_)
print("précision (accuracy) :",svm_cv.best_score_)

## TÂCHE 7

Calculez la précision (accuracy) sur les données de test avec la méthode <code>score</code> :

In [ ]:
svm_cv.score(X_test, Y_test)

Nous pouvons tracer la matrice de confusion

In [ ]:
yhat=svm_cv.predict(X_test)
plot_confusion_matrix(Y_test,yhat)

## TÂCHE 8

Créez un objet arbre de décision, puis créez un objet <code>GridSearchCV</code> nommé <code>tree_cv</code> avec cv = 10. Ajustez (fit) l'objet pour trouver les meilleurs paramètres du dictionnaire <code>parameters</code>.

In [ ]:
parameters = {'criterion': ['gini', 'entropy'],
     'splitter': ['best', 'random'],
     'max_depth': [2*n for n in range(1,10)],
     'max_features': ['sqrt'],  # 'auto' (équivalent à 'sqrt') n'existe plus dans scikit-learn >= 1.3
     'min_samples_leaf': [1, 2, 4],
     'min_samples_split': [2, 5, 10]}

tree = DecisionTreeClassifier()

In [ ]:
tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, Y_train)

In [ ]:
print("hyperparamètres optimisés (meilleurs paramètres) :",tree_cv.best_params_)
print("précision (accuracy) :",tree_cv.best_score_)

## TÂCHE 9

Calculez la précision (accuracy) de tree_cv sur les données de test avec la méthode <code>score</code> :

In [ ]:
tree_cv.score(X_test, Y_test)

Nous pouvons tracer la matrice de confusion

In [ ]:
yhat = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test,yhat)

## TÂCHE 10

Créez un objet k plus proches voisins (KNN), puis créez un objet <code>GridSearchCV</code> nommé <code>knn_cv</code> avec cv = 10. Ajustez (fit) l'objet pour trouver les meilleurs paramètres du dictionnaire <code>parameters</code>.

In [ ]:
parameters = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'p': [1,2]}

KNN = KNeighborsClassifier()

In [ ]:
knn_cv = GridSearchCV(KNN, parameters, cv=10)
knn_cv.fit(X_train, Y_train)

In [ ]:
print("hyperparamètres optimisés (meilleurs paramètres) :",knn_cv.best_params_)
print("précision (accuracy) :",knn_cv.best_score_)

## TÂCHE 11

Calculez la précision (accuracy) de knn_cv sur les données de test avec la méthode <code>score</code> :

In [ ]:
knn_cv.score(X_test, Y_test)

Nous pouvons tracer la matrice de confusion

In [ ]:
yhat = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test,yhat)

## TÂCHE 12

Déterminez la méthode la plus performante :

In [ ]:
# Comparer les quatre méthodes : précision sur les données de test et précision en validation croisée
models = {
    'Régression logistique': logreg_cv,
    'SVM': svm_cv,
    'Arbre de décision': tree_cv,
    'KNN': knn_cv,
}
results = pd.DataFrame({
    'Précision test': {name: m.score(X_test, Y_test) for name, m in models.items()},
    'Précision validation croisée': {name: m.best_score_ for name, m in models.items()},
})
display(results)

best_test = results['Précision test'].max()
best_models = results.index[results['Précision test'] == best_test].tolist()
print("Méthode(s) la/les plus performante(s) sur les données de test :", best_models)
# En cas d'égalité (18 échantillons de test seulement), on départage avec la validation croisée
print("Meilleure selon la validation croisée :", results['Précision validation croisée'].idxmax())

## Auteurs

[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. Tous droits réservés. <h3/>